In [ ]:
import functools
import numpy as np
import qiskit as qk
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm
import qutip as qt
from tqdm import tqdm

import sys
sys.path.append("../")
import pulse_simulator as ps

import matplotlib.pyplot as plt

In [ ]:
def get_closest_multiple_of(value, base_number):
    return int(value + base_number/2) - (int(value + base_number/2) % base_number)


def get_closest_multiple_of_16(num, granularity=16):
    return get_closest_multiple_of(num, granularity)


def qiskit_closest_multiple_of_16(gate_time_si_units, backend):
    with qk.pulse.build(name="dummy", backend=backend) as dummy:
        samples = get_closest_multiple_of_16(
            qk.pulse.seconds_to_samples(gate_time_si_units)
        )
    return samples

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()

dt = backend.configuration().dt * ns
desired_duration = 225 # dt

# Can be acquired with backend.configuration().timing_constraints['granularity']
granularity = 16
GATE_TIME_SI_UNITS = desired_duration * dt / ns
duration = qiskit_closest_multiple_of_16(GATE_TIME_SI_UNITS, backend)

# Pairwise gate simulation
-----

## Custom pulses

### Load from PICO

In [ ]:
pico_a = ps.from_csv("../pico-pulses/saved-pulses-2024-01-24/a_single_qubit_gateset_R1e-3.csv")
fig, ax = plt.subplots(figsize=[5,3])
for row in pico_a:
    ax.plot(row)

In [ ]:
np.trapz(pico_a[0], dx=dt) / np.pi

### Symbolic gates (interpolate with splines)

In [ ]:
# TODO: This is a bit of a hack, but it works for now

def gateset_from_pico(duration, dt, pico_arr, keys, amp=1.0, angle=0.0):
    assert pico_arr.shape[0] == len(keys)

    gate_lookup = {}

    samples = pico_arr.shape[1]
    for k, row in zip(keys, pico_arr):
        signal_samples = ps.resample(
            duration, ps.get_samples(ps.custom_envelope(samples, row, dt=dt))
        )
        gate_lookup[k] = ps.scalable_symbolic_pulse(
            duration, signal_samples, "custom", amp=amp, angle=angle
        )

    return gate_lookup

In [ ]:
pico_gates = {key: row for key, row in zip(["x1", "x2", "sx1", "sx2"], pico_a)}

In [ ]:
# This will take some time at first
gates = gateset_from_pico(duration, dt, pico_a[:2], ["x1", "x2"]) 

In [ ]:
# ... but now it's cached
y_gates = gateset_from_pico(duration, dt, pico_a[:2], ["x1", "x2"], angle=np.pi/2) 

In [ ]:
np.trapz(ps.get_samples(gates["x2"]).real, dx=dt) / np.pi

In [ ]:
samples = ps.get_samples(gates["x1"])

fig, ax = plt.subplots(figsize=(5, 3))
# Rescale to match the original length
scale = pico_a.shape[1] / duration
plt.plot(np.arange(0, duration) * scale, samples.real / scale, label="custom")
plt.plot(np.arange(pico_a.shape[1]), pico_a[0], label="pico", ls=':')
plt.legend()

### Default gates (gaussians)

In [ ]:
default_gates = {
    "x1": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/2),
    "x2": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/2),
    "sx1": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/4),
    "sx2": ps.gaussian_envelope(duration, dt=dt, angle=np.pi/4),
}

# Test
np.trapz(ps.get_samples(default_gates["x1"]).real, dx=dt) / np.pi

## Schedules

TODO: Can we change the samples in the waveform returned by SymbolicPulse?

We could rely on the previous (memoized) interpolation, but I am not sure exactly how to do it (or if it is necessary).

In [ ]:
qubit_sched = {0: "x1", 1: "x2"}
# qubit_sched = {0: "x1"}

before = simpson(ps.get_samples(gates[qubit_sched[1]]).real, dx=dt)
print(f"Before {before / np.pi} π")

with qk.pulse.build(name="Robust moment", backend=backend) as robust_sched:
    for i, key in qubit_sched.items():
        channel = qk.pulse.DriveChannel(i)
        if i == 0:
            qk.pulse.shift_phase(np.pi/2, channel)
        qk.pulse.play(gates[key], channel)

with qk.pulse.build(name="Default moment", backend=backend) as default_sched:
    for i, key in qubit_sched.items():
        channel = qk.pulse.DriveChannel(i)
        if i == 0:
            qk.pulse.shift_phase(np.pi/2, channel)
        qk.pulse.play(default_gates[key], channel)

after = np.trapz(ps.get_samples(robust_sched.instructions[-1][1].pulse), dx=dt)
print(f"After (real) {after.real / np.pi} π")
print(f"After (imag) {after.imag / np.pi} π")

Two options: Circuit or pulse schedule.

In [ ]:
robust_sched.draw()

In [ ]:
default_sched.draw()

## Models

In [ ]:
# Define models fixed by the current parameters

# Backend
# =====
registers = [0, 1]

# NOTE: If the Rabi rates are True, you have to calibrate!
config_vars = ps.backend_simulation_vars(backend, rabi=False, units=Units)

# Carrier frequencies of each control line
carriers = ps.backend_carriers(backend, config_vars)

# =====
H_rx = functools.partial(
    ps.rx_model, 
    registers=registers,
    backend=backend,
    variables=config_vars,
    rotating_frame=False,
)

A_decay = functools.partial(
    ps.qubit_decay_model,
    registers=registers, 
    variables=config_vars
)

H_xtalk = ps.crosstalk_model(
    registers,
    ps.backend_edges(backend),
    config_vars
)

# Create a system model
# =====
# Control model
#   Single qubit circuit moment: attach RX models to each qubit
H_drift = 0.
Hs_control = []
Hs_channels = []
As_static = []
for qubit, label in qubit_sched.items():
    
    # if qubit_index in registers:
    Hj_drift, Hjs_control, Hjs_channel = H_rx(qubit)

    # NOTE: doesn't work with quantum channel simulator yet
    Ajs_static = A_decay(qubit)

    H_drift += Hj_drift
    # Hs_control += Hjs_control
    # Double if RWA is used because 1/2 appears after rotating frame
    Hs_control += [2 * H for H in Hjs_control]
    Hs_channels += Hjs_channel
    As_static += Ajs_static

## Solvers

In [ ]:
# Identity matrix
U0 = ps.qiskit_identity_operator(len(registers))

# # Identity channel
# C0 = qk.quantum_info.SuperOp(U0)

carriers = ps.backend_carriers(backend, config_vars)

In [ ]:
xi_arr = np.linspace(0, 0.01, 10)
# target = ps.from_label("XX")
# target = ps.from_label("YY")
target = ps.from_label("YX")
H_xtalk_norm = H_xtalk / np.max(H_xtalk)

def solve_crosstalk(xi):
    _show_ops = False

    # Get the carriers
    registered_carriers = {c: carriers[c] for c in Hs_channels}
    rcf = 2 * np.mean(list(registered_carriers.values()))

    # Construct the solvers
    solver = qk_d.Solver(
        static_hamiltonian=H_drift.data + xi * np.zeros_like(H_xtalk_norm),
        hamiltonian_operators=Hs_control,
        hamiltonian_channels=Hs_channels,
        rotating_frame=H_drift,
        rwa_cutoff_freq=rcf,
        channel_carrier_freqs=registered_carriers,
        # channel_carrier_freqs={ch: 0. for ch in Hs_channels},
        dt=dt
    )

    solver_xtalk= qk_d.Solver(
        static_hamiltonian=H_drift.data + xi * H_xtalk_norm.data,
        hamiltonian_operators=Hs_control,
        hamiltonian_channels=Hs_channels,
        rotating_frame=H_drift,
        rwa_cutoff_freq=rcf,
        channel_carrier_freqs=registered_carriers,
        # channel_carrier_freqs={ch: 0. for ch in Hs_channels},
        dt=dt
    )

    # Rollout without crosstalk
    # -------------------------
    solver.model.evaluation_mode = 'sparse'
    sol = solver.solve(
        t_span=[0., duration * dt], 
        y0=U0, 
        signals=robust_sched,
        atol=1e-12,
        rtol=1e-12,
        method='jax_odeint'
    )
    sol_default = solver.solve(
        t_span=[0., duration * dt], 
        y0=U0, 
        signals=default_sched,
        atol=1e-12,
        rtol=1e-12,
        method='jax_odeint'
    )

    if _show_ops:
        without_crosstalk = sol.y[-1]
        without_crosstalk_default = sol_default.y[-1]
    else:
        without_crosstalk = qk.quantum_info.process_fidelity(sol.y[-1], target)
        without_crosstalk_default = qk.quantum_info.process_fidelity(sol_default.y[-1], target)

    # Rollout with crosstalk
    # ----------------------
    solver_xtalk.model.evaluation_mode = 'sparse'
    sol_xtalk = solver_xtalk.solve(
        t_span=[0., duration * dt], 
        t_eval=[duration * dt],
        y0=U0, 
        signals=robust_sched,
        atol=1e-16,
        rtol=1e-16,
        method='jax_odeint'
    )
    sol_xtalk_default = solver_xtalk.solve(
        t_span=[0., duration * dt], 
        t_eval=[duration * dt],
        y0=U0, 
        signals=default_sched,
        atol=1e-16,
        rtol=1e-16,
        method='jax_odeint'
    )

    if _show_ops:
        with_crosstalk = sol_xtalk.y[-1]
        with_crosstalk_default = sol_xtalk_default.y[-1]
    else:
        with_crosstalk = qk.quantum_info.process_fidelity(sol_xtalk.y[-1], target)
        with_crosstalk_default = qk.quantum_info.process_fidelity(sol_xtalk_default.y[-1], target)

    return xi, with_crosstalk, without_crosstalk, with_crosstalk_default, without_crosstalk_default

r = []
for xi in tqdm(xi_arr):
    r.append(solve_crosstalk(xi))
results = np.array(r).T

In [ ]:
# Put a floor on the infidelity
infid_floor = np.max(1 - results[1:, 0])
infid = np.where(1 - results[1:] > infid_floor, 1 - results[1:], infid_floor)

fig, ax = plt.subplots()
# xi, with_crosstalk, without_crosstalk, with_crosstalk_default, without_crosstalk_default
ax.plot(results[0], infid[1], lw=2, ls='--', label='without, robust')
ax.plot(results[0], infid[0], lw=2, ls='--', label='with, robust')

ax.plot(results[0], infid[3], label='without, default')
ax.plot(results[0], infid[2], label='with, default')
ax.legend()
# ax.set_ylim([-0.1, 1.1])
ax.set_yscale('log')

# Run calibration
-----

In [ ]:
drive_amp = qk.circuit.Parameter('amp')
drive_amps = np.linspace(0.5, 1.5, 51)

In [ ]:
pico_gates.keys()

## Qiskit dynamics (simulated) calibration

In [ ]:
# Rabi schedules as parameterized pulse instructions
# =====

# Create rabi schedule
qubit = 0
gate_key = "x1"
with qk.pulse.build(name="Rabi moment", backend=backend) as rabi_sched:
    signal = pico_gates[gate_key]
    signal_samples = ps.resample(
        duration, ps.get_samples(ps.custom_envelope(len(signal), signal, dt=dt))
    )
    param_gate = ps.scalable_symbolic_pulse(
        duration, signal_samples, gate_key, amp=drive_amp
    )
    qk.pulse.play(param_gate, qk.pulse.DriveChannel(qubit))

# Assign parameters
exp_rabi_scheds = [rabi_sched.assign_parameters({drive_amp: a}, inplace=False) for a in drive_amps]

In [ ]:
# Backend
# =====
registers = [0]

# NOTE: If the Rabi rates are True, you have to calibrate!
config_vars = ps.backend_simulation_vars(backend, rabi=True, units=Units)

# Carrier frequencies of each control line
carriers = ps.backend_carriers(backend, config_vars)

# =====
H_rx = functools.partial(
    ps.rx_model, 
    registers=registers,
    backend=backend,
    variables=config_vars,
    rotating_frame=True,
)

A_decay = functools.partial(
    ps.qubit_decay_model,
    registers=registers, 
    variables=config_vars
)

H_xtalk = ps.crosstalk_model(
    registers,
    ps.backend_edges(backend),
    config_vars
)

# Create a system model
# =====
# Control model
#   Single qubit circuit moment: attach RX models to each qubit
H_drift = 0.
Hs_control = []
Hs_channels = []
As_static = []
for qubit, label in qubit_sched.items():
    
    # if qubit_index in registers:
    Hj_drift, Hjs_control, Hjs_channel = H_rx(qubit)

    # NOTE: doesn't work with quantum channel simulator yet
    Ajs_static = A_decay(qubit)

    H_drift += Hj_drift
    Hs_control += Hjs_control
    Hs_channels += Hjs_channel
    As_static += Ajs_static

In [ ]:
# Construct solver
solver = qk_d.Solver(
    static_hamiltonian=np.zeros_like(H_xtalk),
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    channel_carrier_freqs={ch: 0. for ch in Hs_channels},
    dt=dt
)

In [ ]:
psi0 = ps.qiskit_ground_state(len(registers))

# Vmap of the rabi calibrations
solver.model.evaluation_mode = 'sparse'
solns = solver.solve(
    t_span=[0., duration * dt], 
    y0=psi0, 
    signals=exp_rabi_scheds,
    atol=1e-16,
    rtol=1e-16,
    method='jax_odeint'
)

rabi_data = np.array([np.abs(s.y[-1][1])**2 for s in solns])

To calibrate, let's just fit to a squared sine.

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
# Define the sine function
def sine_func(x, angle):
    return np.sin(angle * x)**2

# Fit the data to the sine function
x = drive_amps
y = rabi_data

# Initial guess for the parameters is the designed angle
angle_guess = np.trapz(pico_gates[gate_key], dx=dt)
params, _ = curve_fit(sine_func, x, y, p0=[angle_guess])

# Extract the fit parameters
angle_fit, = params

# Calculate the y values for the fit function
x_fit = np.linspace(min(x), max(x), 100)
y_fit = sine_func(x_fit, *params)

# Plot the data and the fit function
plt.plot(x, y, 'o', label='Data')
plt.plot(x_fit, y_fit, label='Fit')
plt.xlabel('Drive Amplitude')
plt.ylabel('Rabi Data')
plt.legend()

In [ ]:
angle_guess / np.pi, angle_fit / np.pi

In [ ]:
# Adjusted amplitude
correction_factor = angle_guess / angle_fit

rabi_coupling = 2 * np.pi * config_vars["omegad0"]

print(f"Correction factor: {correction_factor}")
print(f"Rabi coupling: {rabi_coupling}")
print(f"Check corrected amplitude: {rabi_coupling * correction_factor}")

## Qiskit (circuit-based) calibration

A custom pulse gate cannot be simulated easily. It can be run on IBM hardware.

In [ ]:
# Rabi circuits
# =====

# Add calibratable rabi gate to circuit
rabi_gate = qk.circuit.Gate("rabi", 1, [drive_amp])
qc_rabi = qk.QuantumCircuit(1, 1)
qc_rabi.append(rabi_gate, [0])
qc_rabi.measure(0, 0)

# Register a low-level, custom pulse definition for the given gate.
qc_rabi.add_calibration(rabi_gate, (0,), rabi_sched, [drive_amp])

# Execute rabi circuits
drive_amps = np.linspace(.8, 1.2, 21)
exp_rabi_circs = [qc_rabi.assign_parameters({drive_amp: a}, inplace=False) for a in drive_amps]

In [ ]:
# Check that the parameters have been assigned correctly
exp_rabi_circs[-1].get_instructions("rabi")

In [ ]:
rabi_schedule = qk.schedule(exp_rabi_circs[-1], backend)
rabi_schedule.draw(backend=backend)

In [ ]:
num_shots_per_point = 1024

job = backend.run(
    exp_rabi_circs, meas_level=1, meas_return="avg", shots=num_shots_per_point
)

The next cell fails because fake backends don't have pulse gate support
- What about FakeOpenPulse2Q()? Also doesn't seem to work. 
- ...Plus, it would just do the same thing as the qiskit dynamics example.

In [ ]:
job.result()

# Miscellany
-----

In [ ]:
# Check that resample works.
plt.step(np.linspace(0, duration, samples, endpoint=False), ps.resample(samples, gate_lookup[gate].samples).real)
plt.step(np.arange(0, duration), gate_lookup[gate].samples.real)